# DMS classifier training (Kaggle)

Trains the security-level and document-type classifiers on `dataset.csv` exported by
`ml/export_training_data.py`. **Security reminder:** only the synthetic-safe export
(or a double-gated, explicitly confirmed real export) may ever be uploaded here.
Document text must not leave the deployment outside that gated path.

In [ ]:
!pip install -q -r requirements-kaggle.txt

## 1. Load the exported dataset

Upload `dataset.csv` (Kaggle: *Add Input* -> your dataset). Columns:
`text_excerpt,label_doc_type,label_level,source`.

In [ ]:
import pandas as pd

DATASET_PATH = "/kaggle/input/<your-dataset>/dataset.csv"  # adjust after attaching input
frame = pd.read_csv(DATASET_PATH)
print(frame["source"].value_counts())
frame.head()

## 2. Embed excerpts (self-hosted model weights, runs on the Kaggle VM)

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_ID = "BAAI/bge-small-en-v1.5"
encoder_model = SentenceTransformer(EMBEDDING_MODEL_ID)
X = encoder_model.encode(frame["text_excerpt"].astype(str).str.slice(0, 4000).tolist(),
                         show_progress_bar=True)
assert X.shape[1] == 384, f"dim {X.shape[1]} != contract dim 384"
print(X.shape)

## 3. Train both calibrated heads

Logistic regression with sigmoid calibration; stratified split; per-target label encoders.

In [ ]:
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGETS = (("doc_type", "label_doc_type"), ("security_level", "label_level"))
sources = frame["source"].astype(str).to_numpy()

trained = {}
for target, column in TARGETS:
    le = LabelEncoder()
    y = le.fit_transform(frame[column].astype(str).to_numpy())
    try:
        train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE, stratify=y)
    except ValueError:
        print(f"{target}: classes too small for stratification; unstratified split")
        train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE)
    clf = CalibratedClassifierCV(
        LogisticRegression(max_iter=1000, class_weight="balanced"), method="sigmoid", cv=5)
    clf.fit(X[train_idx], y[train_idx])
    trained[target] = {"model": clf, "label_encoder": le,
                       "test_idx": test_idx, "y_test": y[test_idx],
                       "y_pred": clf.predict(X[test_idx])}
    print(f"trained {target} on {len(train_idx)} rows ({len(le.classes_)} classes)")

## 4. Metrics per source slice -- per-class recall, never a lone accuracy number

Synthetic and real slices are reported separately; the highest security label
(`Restricted`) recall is called out explicitly and must stay near 1.0.

In [ ]:
from sklearn.metrics import recall_score

metrics = {}
for target, _ in TARGETS:
    run = trained[target]
    src_test = sources[run["test_idx"]]
    classes = run["label_encoder"].classes_
    metrics[target] = {}
    for slice_name in ("synthetic", "real"):
        mask = src_test == slice_name
        if not mask.any():
            metrics[target][slice_name] = None
            print(f"[{target}/{slice_name}] no rows -> null")
            continue
        values = recall_score(run["y_test"][mask], run["y_pred"][mask],
                              labels=classes, average=None, zero_division=0)
        per_class = {str(c): round(float(v), 4) for c, v in zip(classes, values, strict=True)}
        metrics[target][slice_name] = {"support": int(mask.sum()),
                                       "per_class_recall": per_class,
                                       "restricted_recall": per_class.get("Restricted")}
        print(f"[{target}/{slice_name}] support={int(mask.sum())}")
        for label, recall in sorted(per_class.items()):
            print(f"    recall[{label}] = {recall}")
        if per_class.get("Restricted") is not None:
            print(f"    restricted_recall = {per_class['Restricted']}  <- highest-label gate")

## 5. Save the artifact (`model.joblib` + `metrics.json`)

Schema v1 per `ml/artifact_contract.md`: sklearn major.minor must match the backend,
embedding dim fixed at 384, labels subset of the taxonomy.

In [ ]:
import json

import joblib
import sklearn

manifest = {
    "schema_version": 1,
    "sklearn_version": sklearn.__version__,
    "embedding_model_id": EMBEDDING_MODEL_ID,
    "dim": int(X.shape[1]),
    "labels": {t: sorted(str(c) for c in trained[t]["label_encoder"].classes_)
               for t, _ in TARGETS},
    "metrics": metrics,
}
artifact = {"manifest": manifest,
            "models": {t: {"model": trained[t]["model"],
                           "label_encoder": trained[t]["label_encoder"]} for t, _ in TARGETS}}
joblib.dump(artifact, "model.joblib")
with open("metrics.json", "w", encoding="utf-8") as fh:
    json.dump(manifest["metrics"], fh, indent=2, sort_keys=True)
print("wrote model.joblib + metrics.json -- download both from the output panel")